# Week 15: Graded Mini Project — Domain-Specific RAG Conversational Assistant

**Course**: IITM Pravartak — Agentic AI and Applications  
**Student**: Maharshi  
**Domain**: E-Commerce (ShopEase) — Product Policies & Customer Support  
**Framework**: LangChain + OpenAI API + FAISS Vector Store  

---

## Project Overview
This project implements a complete, production-grade **Retrieval-Augmented Generation (RAG) conversational assistant** for an e-commerce platform (**ShopEase**). The assistant answers customer questions regarding returns, refunds, shipping, product warranties, payment methods, and prime memberships strictly based on indexed policy documents.

### Key Features
1. **Document Ingestion**: Multi-document loader with semantic recursive chunking
2. **Vector Indexing**: OpenAI `text-embedding-3-small` embeddings stored in a local FAISS index
3. **Conversational RAG**: `ConversationalRetrievalChain` with sliding-window memory for multi-turn follow-ups
4. **Strict Grounding & Anti-Hallucination**: Dedicated system prompt enforcing explicit document citation and refusal when information is absent
5. **Source Attribution**: Transparent document-level citation in all responses

## Setup & Environment Configuration
Install required packages and configure environment variables securely without hardcoded secrets.

In [1]:
# Environment setup and imports
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv('.env')

api_key = os.environ.get('OPENAI_API_KEY')
api_base = os.environ.get('OPENAI_API_BASE')
model_name = os.environ.get('MODEL', 'gpt-4o-mini')

print(f"API Key configured: {'Yes (...' + api_key[-6:] + ')' if api_key else 'No'}")
print(f"API Endpoint: {api_base if api_base else 'Default OpenAI'}")
print(f"Model: {model_name}")

API Key configured: Yes (...262617)
API Endpoint: https://openai.vocareum.com/v1
Model: gpt-4o-mini


## Section A: Domain Selection & Document Ingestion

### Domain Selection
We selected the **E-Commerce** domain. Five publicly inspired policy documents are loaded from the `documents/` directory:
1. `return_policy.txt`: 30-day return window, non-returnable categories, refund timelines, electronics return policies
2. `shipping_policy.txt`: Standard, express, and same-day delivery tiers, fees, pin code rules
3. `warranty_policy.txt`: Manufacturer warranty, ShopEase Assurance, Protect+ accidental damage plans
4. `payment_faq.txt`: Cards, UPI, Net Banking, COD limits, EMI options, wallet rules
5. `membership_program.txt`: Prime tiers (Monthly, Annual, Student, Family), shipping benefits, discounts

In [2]:
from langchain_community.document_loaders import TextLoader

documents_dir = Path('documents')
raw_documents = []

for doc_path in sorted(documents_dir.glob('*.txt')):
    loader = TextLoader(str(doc_path), encoding='utf-8')
    loaded_docs = loader.load()
    for d in loaded_docs:
        d.metadata['source'] = doc_path.name
    raw_documents.extend(loaded_docs)
    print(f"Loaded '{doc_path.name}': {len(loaded_docs[0].page_content)} characters")

print(f"\nTotal documents ingested: {len(raw_documents)}")

Loaded 'membership_program.txt': 4189 characters
Loaded 'payment_faq.txt': 4299 characters
Loaded 'return_policy.txt': 4554 characters
Loaded 'shipping_policy.txt': 4841 characters
Loaded 'warranty_policy.txt': 3983 characters

Total documents ingested: 5


## Section B: Semantic Chunking & FAISS Vector Store

We split documents using `RecursiveCharacterTextSplitter` with:
- **Chunk Size**: `800` characters (preserves complete policy clauses)
- **Chunk Overlap**: `150` characters (ensures semantic continuity across chunk boundaries)
- **Embeddings**: OpenAI `text-embedding-3-small`
- **Vector Store**: FAISS (Facebook AI Similarity Search) persisted to disk

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Step 1: Semantic Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', ''],
)
chunks = text_splitter.split_documents(raw_documents)
print(f"Generated {len(chunks)} chunks from {len(raw_documents)} documents.")

# Step 2: Embedding Generation & FAISS Index Creation
embedding_kwargs = {'model': 'text-embedding-3-small', 'openai_api_key': api_key}
if api_base:
    embedding_kwargs['openai_api_base'] = api_base

embeddings = OpenAIEmbeddings(**embedding_kwargs)
vectorstore = FAISS.from_documents(chunks, embeddings)

# Step 3: Persist to disk
vectorstore_dir = Path('vectorstore')
vectorstore_dir.mkdir(exist_ok=True)
vectorstore.save_local(str(vectorstore_dir))
print(f"FAISS index saved to {vectorstore_dir}/ with {vectorstore.index.ntotal} vectors.")

Generated 34 chunks from 5 documents.


FAISS index saved to vectorstore/ with 34 vectors.


In [4]:
# Verification of vector search retrieval
sample_query = "What is the return window for electronics?"
retrieved = vectorstore.similarity_search(sample_query, k=2)
print(f"Query: '{sample_query}'\n")
for idx, r in enumerate(retrieved, 1):
    print(f"[{idx}] Source: {r.metadata['source']}")
    print(f"Content snippet: {r.page_content[:150].strip()}...\n")

Query: 'What is the return window for electronics?'

[1] Source: return_policy.txt
Content snippet: 3. EXCHANGE POLICY

3.1 Direct Exchanges
- Size and color exchanges are available within 30 days
- Exchange items must be of equal or lesser value
- I...

[2] Source: return_policy.txt
Content snippet: 4.3 Damaged Electronics
- If the product arrives damaged, report within 48 hours of delivery
- Include photographic evidence of damage
- A replacement...



## Section C: Conversational RAG Pipeline with Grounded Prompting

The pipeline coordinates:
1. **Condense Question Step**: Translates contextual follow-ups into standalone search queries based on chat history.
2. **Similarity Retriever**: Extracts top-k (k=4) relevant document chunks.
3. **Grounded Prompting**: System prompt strictly forbids using outside knowledge and requires explicit refusal if absent.
4. **Sliding Window Memory**: Retains the last 8 conversation turns.

In [5]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.prompts import PromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.prompts.chat import ChatPromptTemplate

SYSTEM_PROMPT = """You are ShopEase Support Assistant, a helpful and accurate customer support chatbot for ShopEase E-Commerce.

STRICT RULES:
1. Answer ONLY based on the provided context documents. Do NOT use your general knowledge.
2. If the answer is not found in the provided context, respond with: "I don't have enough information in the provided documents to answer that question. Please contact our support team at 1800-SHOP-EASE for further assistance."
3. When answering, cite the specific policy or document section (e.g., "According to our Return Policy (Section 4.1)...").
4. Be professional, helpful, and empathetic.
5. For follow-up questions, use the conversation history to maintain context.
6. If the question is ambiguous, ask for clarification.
7. Provide specific details like timelines, costs, and steps when available in the documents.
8. Never promise anything that is not explicitly stated in the policy documents.

CONTEXT FROM DOCUMENTS:
{context}
"""

CONDENSE_PROMPT = """Given the following conversation history and a follow-up question, rephrase the follow-up question to be a standalone question that captures the full context.

Chat History:
{chat_history}

Follow-up Question: {question}

Standalone Question:"""

# Initialize LLM
llm_kwargs = {'model': model_name, 'temperature': 0.1, 'openai_api_key': api_key}
if api_base:
    llm_kwargs['openai_api_base'] = api_base
llm = ChatOpenAI(**llm_kwargs)

# Retriever & Memory
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 4})
memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    return_messages=True,
    output_key='answer',
    k=8
)

# Prompts
qa_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template('{question}'),
])
condense_prompt = PromptTemplate.from_template(CONDENSE_PROMPT)

# Build Chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    condense_question_prompt=condense_prompt,
    combine_docs_chain_kwargs={'prompt': qa_prompt},
    return_source_documents=True,
    verbose=False,
)

print("Conversational RAG Chain initialized successfully.")

Conversational RAG Chain initialized successfully.


C:\Users\Maharshi\AppData\Local\Temp\ipykernel_26964\3418694741.py:40: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


## Section D: Evaluation, Multi-Turn Follow-Ups & Hallucination Refusal

We evaluate the chatbot across four rigorous dimensions:
1. **Direct Fact Retrieval**: Verifies accuracy of retrieved policy thresholds, prices, and rules
2. **Conversational Memory & Follow-Ups**: Validates resolution of pronouns ("What if it arrived damaged?", "How long do I have to report it?")
3. **Cross-Document Queries**: Combines shipping, payment, warranty, and membership policies
4. **Out-of-Scope / Anti-Hallucination Test**: Verifies strict refusal when asking for information outside the document corpus

In [6]:
# Helper execution function
def ask_bot(question: str):
    print(f"\n{'='*70}")
    print(f"User: {question}")
    print(f"{'='*70}")
    
    result = rag_chain.invoke({'question': question})
    answer = result['answer']
    sources = list(set(d.metadata['source'] for d in result['source_documents']))
    
    print(f"\nAssistant: {answer}")
    print(f"\nCitations: {', '.join(sources)}")
    return answer, sources

In [7]:
# Test Case 1: Direct Return Policy Question
ask_bot("What is ShopEase's return policy for electronics?");


User: What is ShopEase's return policy for electronics?



Assistant: According to our Return Policy (Section 4.1), electronics have a 15-day return window, which is shorter than the standard 30 days for most items. The return requirements for electronics include:

- All original accessories, cables, manuals, and packaging must be included.
- The device must be factory reset before return.
- The serial number must match the one on the original invoice.
- Activated software licenses cannot be returned.

If you have any further questions or need assistance with a return, please let me know!

Citations: warranty_policy.txt, return_policy.txt


In [8]:
# Test Case 2: Multi-turn Follow-up 1 (Pronoun resolution & context awareness)
ask_bot("What if the electronics item arrived damaged?");


User: What if the electronics item arrived damaged?



Assistant: According to our Damaged Electronics policy (Section 4.3), if the product arrives damaged, you should report it within 48 hours of delivery. You will need to include photographic evidence of the damage. Once reported, a replacement or full refund will be processed within 2 business days.

Citations: return_policy.txt, shipping_policy.txt


In [9]:
# Test Case 3: Multi-turn Follow-up 2 (Deeper contextual dependency)
ask_bot("How long do I have to report the damage?");


User: How long do I have to report the damage?



Assistant: According to our Damaged Electronics policy (Section 4.3), you must report a damaged electronics item within 48 hours of delivery.

Citations: warranty_policy.txt, return_policy.txt, shipping_policy.txt


In [10]:
# Test Case 4: Shipping Policy & City Restrictions
ask_bot("What are the shipping options, costs, and is same-day delivery available in Pune?");


User: What are the shipping options, costs, and is same-day delivery available in Pune?



Assistant: In Pune, the available shipping options and costs are as follows:

1. **Same-Day Delivery**:
   - Delivery Time: Same day for orders placed before 12:00 PM.
   - Cost: INR 199.
   - Note: Same-day delivery is available only for items from local sellers, not for items shipped from the warehouse.

2. **Standard Shipping**:
   - Delivery Time: 5-7 business days.
   - Cost: Free for orders above INR 499; INR 49 for orders below INR 499.

3. **Express Shipping**:
   - Delivery Time: 2-3 business days.
   - Cost: INR 99 for orders above INR 499; INR 149 for orders below INR 499.

Additionally, you can choose scheduled delivery for a preferred date and time slot, which costs INR 79 on top of standard shipping and must be scheduled at least 2 days in advance.

If you have any further questions or need assistance, feel free to ask!

Citations: membership_program.txt, shipping_policy.txt


In [11]:
# Test Case 5: Warranty & Accidental Damage Coverage
ask_bot("What does the ShopEase Protect+ Complete plan cover and what is the deductible?");


User: What does the ShopEase Protect+ Complete plan cover and what is the deductible?



Assistant: According to the Extended Warranty (ShopEase Protect+) section, the Protect+ Complete plan covers accidental damage for 2 years. This includes coverage for drops, spills, screen cracks, and electrical surges. 

The deductible amounts for claims under the Protect+ Complete plan are as follows:
- INR 500 per claim for items under INR 10,000
- INR 1,000 per claim for items between INR 10,000 and 50,000
- INR 2,500 per claim for items above INR 50,000

Additionally, there is a maximum of 2 claims allowed per year.

Citations: warranty_policy.txt, membership_program.txt


In [12]:
# Test Case 6: Payment Methods & EMI
ask_bot("Can I pay using EMI and what is the minimum order value?");


User: Can I pay using EMI and what is the minimum order value?



Assistant: According to our Payment Methods document (Section 1.6), the options for paying using EMI (Equated Monthly Installments) include:

- Available on credit cards from select banks.
- No-Cost EMI available on select products for 3, 6, 9, or 12 months.
- Standard EMI with interest rates as per the card issuer (typically 12-18% per annum).
- Bajaj Finserv EMI card is also accepted.

The minimum order value required to qualify for this payment method is INR 3,000.

Citations: payment_faq.txt


In [13]:
# Test Case 7: Prime Membership & Student Tier Follow-up
ask_bot("What are the Prime membership tiers and how much is the student plan?");


User: What are the Prime membership tiers and how much is the student plan?



Assistant: According to the ShopEase Prime Membership Program (Section 1.1), the different tiers of Prime membership are as follows:

- **Prime Monthly**: INR 149/month
- **Prime Annual**: INR 999/year (save INR 789 compared to monthly)
- **Prime Student**: INR 499/year (valid student ID required)
- **Prime Family**: INR 1,499/year (up to 4 family members)

The cost of the student plan is INR 499 per year.

Citations: membership_program.txt


In [14]:
# Test Case 8: STRICT REFUSAL / ANTI-HALLUCINATION TEST (Out-of-Scope Query)
# Expected: Refuses to answer and provides customer support number
ask_bot("What is the weather forecast for tomorrow in Mumbai?");


User: What is the weather forecast for tomorrow in Mumbai?



Assistant: I don't have enough information in the provided documents to answer that question. Please contact our support team at 1800-SHOP-EASE for further assistance.

Citations: warranty_policy.txt, payment_faq.txt, shipping_policy.txt


## Summary & Architecture Conclusions

| Requirement | Implementation Detail | Status |
|---|---|---|
| **Non-HR Domain** | E-Commerce Customer Support (ShopEase Policies) | Verified |
| **Public Document Ingestion** | 5 structured policy documents loaded & parsed | Verified |
| **Semantic Chunking** | `RecursiveCharacterTextSplitter` (800 chunk, 150 overlap) | Verified |
| **OpenAI Embeddings** | `text-embedding-3-small` | Verified |
| **Vector Store** | FAISS local indexing with persistence | Verified |
| **Conversational Memory** | `ConversationBufferWindowMemory` (8 turns) + question condenser | Verified |
| **Follow-up Handling** | Seamless pronoun and contextual continuity | Verified |
| **Anti-Hallucination / Refusal** | Strict prompt guarding with explicit refusal statement | Verified |
| **Source Citations** | Document metadata tracking & citation output | Verified |

### Deliverables Included in Submission
- `ingest.py`: Modular ingestion, chunking, embedding, and FAISS indexing pipeline
- `chatbot.py`: Full conversational RAG pipeline with interactive CLI and `--demo` runner
- `app.py`: Streamlit web UI with chat interface and citation expanders (Bonus)
- `README.md`: Architecture diagrams, public source citations, and setup instructions
- `sample_conversation_log.txt`: 10-turn conversation log with ground-truth verification
- `Week_15_Graded_Mini_Project.ipynb`: Self-contained interactive notebook